In [1]:
import random
import time
from SPARQLWrapper import SPARQLWrapper, JSON

# Initialize the SPARQL endpoint for Wikidata
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", "MyAppName/1.0 (belo.fede@outlook.com)")


def fetch_range(min_val, max_val, limit=1000):
    # Adjust the SUBSTR offset (here we use 32) based on the exact prefix length.
    query = f"""
    SELECT DISTINCT ?class ?parent WHERE {{
      ?class wdt:P279 ?parent .
      FILTER(
        xsd:integer(SUBSTR(STR(?class), 32)) >= {min_val} &&
        xsd:integer(SUBSTR(STR(?class), 32)) < {max_val}
      )
    }} LIMIT {limit}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()


ontology = {}
min_bound = 1
max_bound = 200000  # Adjust according to the data you expect
step = 10000

for lower in range(min_bound, max_bound, step):
    upper = lower + step
    print(f"Fetching QIDs with numeric part in range [{lower}, {upper})...")
    result = fetch_range(lower, upper)
    bindings = result["results"]["bindings"]
    if not bindings:
        continue
    for res in bindings:
        class_uri = res["class"]["value"]
        parent_uri = res["parent"]["value"]
        # Extract QIDs (e.g., "Q12345") from the IRI
        class_qid = class_uri.rsplit("/", 1)[-1]
        parent_qid = parent_uri.rsplit("/", 1)[-1]
        if not (class_qid.startswith("Q") and parent_qid.startswith("Q")):
            continue
        # The dictionary deduplicates any overlaps that might still occur.
        if class_qid not in ontology:
            ontology[class_qid] = {parent_qid}
        else:
            if parent_qid not in ontology[class_qid]:
                ontology[class_qid].add(parent_qid)
            else:
                print(f"Duplicate entry: {class_qid} -> {parent_qid}")
    time.sleep(random.random())

print("Total ontology entries retrieved:", len(ontology))

Fetching QIDs with numeric part in range [1, 10001)...


EndPointInternalError: EndPointInternalError: The endpoint returned the HTTP status code 500. 

Response:
b'SPARQL-QUERY: queryStr=\n    SELECT DISTINCT ?class ?parent WHERE {\n      ?class wdt:P279 ?parent .\n      FILTER(\n        xsd:integer(SUBSTR(STR(?class), 32)) >= 1 &&\n        xsd:integer(SUBSTR(STR(?class), 32)) < 10001\n      )\n    } LIMIT 1000\n    \njava.util.concurrent.TimeoutException\n\tat java.util.concurrent.FutureTask.get(FutureTask.java:205)\n\tat com.bigdata.rdf.sail.webapp.BigdataServlet.submitApiTask(BigdataServlet.java:292)\n\tat com.bigdata.rdf.sail.webapp.QueryServlet.doSparqlQuery(QueryServlet.java:678)\n\tat com.bigdata.rdf.sail.webapp.QueryServlet.doGet(QueryServlet.java:290)\n\tat com.bigdata.rdf.sail.webapp.RESTServlet.doGet(RESTServlet.java:240)\n\tat com.bigdata.rdf.sail.webapp.MultiTenancyServlet.doGet(MultiTenancyServlet.java:273)\n\tat javax.servlet.http.HttpServlet.service(HttpServlet.java:687)\n\tat javax.servlet.http.HttpServlet.service(HttpServl

In [2]:
len(ontology)

14859

In [3]:
sorted(
    [(child, parents) for child, parents in ontology.items() if len(parents) > 1],
    key=lambda x: int(x[0][1:]),
    reverse=True,
)

[('Q132631947', {'Q5891', 'Q799'}),
 ('Q132418782', {'Q333', 'Q7108'}),
 ('Q132186465', {'Q413', 'Q431'}),
 ('Q131869953', {'Q3966', 'Q5300'}),
 ('Q131625744', {'Q1827', 'Q282'}),
 ('Q128807812', {'Q3133', 'Q943'}),
 ('Q125684839', {'Q5119', 'Q515'}),
 ('Q125229688', {'Q142', 'Q5'}),
 ('Q124854763', {'Q344', 'Q575'}),
 ('Q124154087', {'Q6235', 'Q7169'}),
 ('Q123924129', {'Q3957', 'Q515'}),
 ('Q123460022', {'Q521', 'Q7168'}),
 ('Q120041664', {'Q3957', 'Q5084', 'Q515', 'Q532'}),
 ('Q118652592', {'Q154', 'Q44'}),
 ('Q118496643', {'Q1568', 'Q1617'}),
 ('Q116742203', {'Q290', 'Q729'}),
 ('Q115864634', {'Q2329', 'Q413'}),
 ('Q115861523', {'Q336', 'Q413'}),
 ('Q115857134', {'Q3465', 'Q5'}),
 ('Q115341608', {'Q5638', 'Q5639'}),
 ('Q115154781', {'Q6689', 'Q676'}),
 ('Q113517952', {'Q344', 'Q5151'}),
 ('Q113160617', {'Q5084', 'Q532'}),
 ('Q113133333', {'Q441', 'Q7141'}),
 ('Q112956198', {'Q521', 'Q7094'}),
 ('Q112939936', {'Q431', 'Q7141'}),
 ('Q111836713', {'Q282', 'Q44'}),
 ('Q111679581', {'Q5